**Exercise 1 - Prompt Chaining for a Customer Support AI**

**Step 1: Issue Clarification**

**Prompt Used:** You are a professional customer support classifier for a SaaS company.

Classify customer messages into ONE of these categories:
- Billing
- Technical Issue
- Account Access
- Feature Request
- Other

Respond ONLY in this JSON format:
{
  "category": "",
  "confidence": "low/medium/high",
  "reason": ""
}

Do not provide explanations outside the JSON.

Customer message:
"I was charged twice this month and I can't log into my account."

**What step does:** Step 1 classifies the customer’s issue into a single structured category.
It produces structured JSON output (category + confidence) that will be used as input for Step 2 and Step 3.

**Output:** {
"category": "Billing",
"confidence": "high",
"reason": "The primary issue mentioned is being charged twice, which falls under billing, even though login access is also noted."
}

**Step 2: Request Missing Information**

**Prompt Used:** You are a customer support assistant.

The issue has been classified as:
Category: Billing
Confidence: high

If important information is missing, ask up to 3 concise clarification questions.

Rules:
- Be professional and polite.
- Do NOT solve the issue yet.
- Do NOT repeat the original message.
- Keep response under 100 words.

**What this step does:** Step 2 uses the classification from Step 1 to determine what additional details are needed. It depends directly on Step 1’s category and confidence to generate targeted clarification questions.

**Output:**

Category: Billing
Confidence: High

Thank you for reaching out. To assist you further, could you please provide:

The dates and amounts of the duplicate charges (and last 4 digits of the card used)?

A screenshot or exact error message shown when attempting to log in?

The email address associated with your account?

Once we have this information, we can review your case more efficiently.

**Step 3: Solution Generator**

**Prompt Used:**
You are a Tier 1 customer support agent.

Based on:
Category: Billing
Customer message: "I was charged twice this month and I can't log into my account."

Provide:
1. A brief explanation of the issue.
2. A clear step-by-step solution.
3. A professional closing line.

Constraints:
- Tone: calm and reassuring
- Length: under 150 words
- Avoid technical jargon

**What This Step Does:** Step 3 generates a tailored solution using the classification from Step 1 and the original message.
It depends on the structured category output to produce a focused and relevant resolution.

**Output:**

It looks like you’re dealing with two problems: a duplicate charge on your subscription and trouble signing in to your account.

Step-by-step:

Check your bank/credit card activity to confirm both charges are “posted” (not pending).

In your billing page, review your invoices/receipts for this month to see if there are two separate invoices.

If you see two posted charges or two invoices, reply with the charge dates, amounts, and the last 4 digits of your card so we can investigate and fix it.

For login: use “Forgot password” on the sign-in page and reset your password.

If you don’t receive the reset email, check spam/junk, then try again after a few minutes.

We’ll get this sorted quickly—reply with the details above and we’ll take the next steps.

**Step 4: Escalation Decision**

**Prompt:**

You are a support escalation decision system.

Given:
Category: Billing
Confidence: High
Message: "I was charged twice this month and I can't log into my account."

Return only this JSON:
{
  "escalate": true/false,
  "reason": ""
}

Escalate if:
- Billing disputes exceed $100
- Security concerns are mentioned
- Confidence level is low

**What This Step Does:** Step 4 determines whether the issue should be escalated to Tier 2 support.
It directly depends on the classification and confidence score from Step 1 to make a structured escalation decision.

**Output:**
{
"escalate": false,
"reason": "No charge amount over $100 is specified, no security concerns are mentioned, and classification confidence is high."
}


**Iteration Log — Before & After Prompt Refinement**

**Problem observed during testing:**
The model sometimes returned multiple categories (e.g., both "Billing" and "Account Access") instead of choosing one, which broke the downstream chain steps that expected a single category.

** Before (original prompt):**

You are a customer support classifier.
Classify this customer message into relevant categories.
Return a JSON with the category and your confidence.

Customer message: "I was charged twice this month and I can't log into my account."


**Broken output:**
json
{
  "categories": ["Billing", "Account Access"],
  "confidence": "high"
}


**After:**

You are a professional customer support classifier for a SaaS company.

Classify customer messages into ONE of these categories:
- Billing
- Technical Issue
- Account Access
- Feature Request
- Other

Respond ONLY in this JSON format:
{
  "category": "",
  "confidence": "low/medium/high",
  "reason": ""
}

Do not provide explanations outside the JSON.


**What changed:**
- Added "ONE of these categories" which prevents multi-label output
- Added "Respond ONLY in this JSON format", removes prose explanations outside JSON
- Added explicit category list, constrains model to valid options only
- Added reason field,  makes classification auditable for downstream steps

**Result:** Model now consistently returns a single category in valid JSON, enabling reliable chaining into Steps 2, 3, and 4.
